# 🏋️‍♂️ Water Quality Model Training

## Purpose
Trains a water quality classification model and registers it in Unity Catalog as **Challenger**

## Steps
1. Load data from Unity Catalog table
2. Engineer features for water quality prediction
3. Train Random Forest classifier
4. Log metrics and model to MLflow
5. Find best run by accuracy
6. Register model in Unity Catalog with **Challenger** alias

## Notes
- Model is registered as **Challenger** (candidate model not yet validated)
- Model deployment workflow will evaluate and potentially promote to **Champion** 
- Follows Iris MLOps pattern for model lifecycle management

In [ ]:
# 📦 Install required packages
%pip install mlflow scikit-learn pandas numpy matplotlib seaborn

# Restart Python 
dbutils.library.restartPython()

In [ ]:
# 📚 Import Libraries
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, roc_auc_score
from mlflow.models.signature import infer_signature
from pyspark.sql import SparkSession
import warnings
warnings.filterwarnings('ignore')

# Initialize Spark and MLflow
spark = SparkSession.builder.getOrCreate()
mlflow.set_registry_uri("databricks-uc")

print("✅ Libraries imported successfully!")

In [ ]:
# 🎯 Configuration - Get from Bundle Parameters
catalog_name = spark.conf.get("catalog_name", "dev_digital_engineering_services")
schema_name = spark.conf.get("schema_name", "default")
model_name = spark.conf.get("model_name", "water_quality_model")
experiment_path = spark.conf.get("experiment_path", "/Shared/water_quality_exp")

# Construct full names
FULL_MODEL_NAME = f"{catalog_name}.{schema_name}.{model_name}"
TABLE_NAME = f"{catalog_name}.{schema_name}.water_quality_data"
EXPERIMENT_NAME = experiment_path

# Set MLflow experiment
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"🎯 Configuration:")
print(f"   📊 Data Table: {TABLE_NAME}")
print(f"   🤖 Model: {FULL_MODEL_NAME}")
print(f"   🔬 Experiment: {EXPERIMENT_NAME}")

In [ ]:
# 📊 Load Data from Unity Catalog
print(f"📊 Loading data from Unity Catalog table: {TABLE_NAME}")

# Load data using Spark
water_spark_df = spark.sql(f"SELECT * FROM {TABLE_NAME}")

# Convert to Pandas for ML processing
water_df = water_spark_df.toPandas()

print(f"✅ Loaded {len(water_df)} samples")
print(f"📋 Columns: {list(water_df.columns)}")

# Show basic info
print(f"\n📈 Dataset Summary:")
print(f"   Shape: {water_df.shape}")
print(f"   Target distribution:")
target_dist = water_df['Potability'].value_counts()
for key, value in target_dist.items():
    print(f"      {key}: {value} ({value/len(water_df)*100:.1f}%)")

In [ ]:
# 🔧 Feature Engineering (Same as established pipeline)
print("🔧 Engineering features for water quality prediction...")

# Core water quality features (from original pipeline)
feature_cols = ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 
               'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']

# Engineered features (MUST match production inference)
water_df['ph_normalized'] = water_df['ph'] / 14
water_df['hardness_ratio'] = water_df['Hardness'] / (water_df['Solids'] + 1)
water_df['organic_load'] = water_df['Organic_carbon'] * water_df['Chloramines']

# Water quality index (composite feature)
water_df['water_quality_index'] = (
    water_df['ph_normalized'] + 
    (1 / (water_df['Turbidity'] + 1)) + 
    (1 / (water_df['hardness_ratio'] + 1))
) / 3

# Final feature set (13 features total)
final_features = feature_cols + ['ph_normalized', 'hardness_ratio', 'organic_load', 'water_quality_index']

print(f"✅ Feature engineering complete")
print(f"📊 Final features ({len(final_features)}): {final_features}")

In [ ]:
# 🎯 Prepare Training Data
# Features and target
X = water_df[final_features]
y = water_df['Potability']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📊 Training Data Prepared:")
print(f"   Training samples: {len(X_train)}")
print(f"   Test samples: {len(X_test)}")  
print(f"   Features: {len(final_features)}")

# Class distribution in train set
train_dist = y_train.value_counts()
print(f"   Training distribution:")
for key, value in train_dist.items():
    print(f"      Class {key}: {value} ({value/len(y_train)*100:.1f}%)")

In [ ]:
# 🏋️‍♂️ Train Random Forest Model with MLflow Tracking
print("🏋️‍♂️ Training Random Forest classifier...")

with mlflow.start_run(run_name="water_quality_training_challenger") as run:
    
    # Model hyperparameters
    n_estimators = 100
    max_depth = 10
    random_state = 42
    
    # Log parameters
    mlflow.log_param("algorithm", "RandomForest")
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("n_features", len(final_features))
    mlflow.log_param("train_samples", len(X_train))
    mlflow.log_param("test_samples", len(X_test))
    
    # Train model
    rf_model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state,
        n_jobs=-1
    )
    
    rf_model.fit(X_train, y_train)
    
    # Predictions
    y_pred = rf_model.predict(X_test)
    y_pred_proba = rf_model.predict_proba(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    roc_auc = roc_auc_score(y_test, y_pred_proba[:, 1])
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    
    print(f"✅ Model Training Complete!")
    print(f"📊 Performance Metrics:")
    print(f"   Accuracy:  {accuracy:.4f}")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall:    {recall:.4f}")
    print(f"   F1 Score:  {f1:.4f}")
    print(f"   ROC AUC:   {roc_auc:.4f}")
    
    run_id = run.info.run_id
    print(f"🔍 MLflow Run ID: {run_id}")

In [ ]:
# 📝 Log Model with Signature
print("📝 Logging model to MLflow...")

with mlflow.start_run(run_id=run_id):
    # Create model signature
    signature = infer_signature(X_train, y_pred)
    
    # Log the model
    mlflow.sklearn.log_model(
        sk_model=rf_model,
        artifact_path="water_quality_model",
        signature=signature,
        input_example=X_train.head(5),
        registered_model_name=FULL_MODEL_NAME
    )
    
    # Log feature importance
    feature_importance = pd.DataFrame({
        'feature': final_features,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    mlflow.log_table(feature_importance, "feature_importance.json")
    
    print("✅ Model logged to MLflow with signature and example")
    print("📊 Top 5 Important Features:")
    print(feature_importance.head())

In [ ]:
# 🎯 Register Model as CHALLENGER in Unity Catalog
print("🎯 Registering model as CHALLENGER in Unity Catalog...")

# Get the latest model version
client = mlflow.tracking.MlflowClient()
latest_version = client.get_latest_versions(FULL_MODEL_NAME, stages=["None"])[0]

# Set alias to CHALLENGER (following Iris pattern)
client.set_registered_model_alias(
    name=FULL_MODEL_NAME,
    alias="Challenger",
    version=latest_version.version
)

# Add tags for tracking
client.set_model_version_tag(
    name=FULL_MODEL_NAME,
    version=latest_version.version,
    key="stage",
    value="challenger"
)

client.set_model_version_tag(
    name=FULL_MODEL_NAME,
    version=latest_version.version,
    key="validation_status", 
    value="pending"
)

print(f"✅ Model registered as CHALLENGER")
print(f"🔍 Model: {FULL_MODEL_NAME}")
print(f"📦 Version: {latest_version.version}")
print(f"🏷️ Alias: Challenger")

print(f"\n🎯 TRAINING COMPLETE!")
print(f"Next: Run model deployment workflow to:")
print(f"  1. Evaluate Challenger model")
print(f"  2. Check for approval")
print(f"  3. Promote to Champion and deploy")